In [16]:
# IMPORTAR LIBRERÍAS NECESARIAS

# numpy: librería matemática por excelencia para manejar los datos numéricos.
import numpy as np

# matplotlib.pyplot: para hacer las gráficas (plots).
import matplotlib.pyplot as plt

# pathlib: para manejar rutas de carpetas y archivos. 
from pathlib import Path

# scipy.io: para abrir archivos de MATLAB
import scipy.io as sio

# pandas: para manejar datos en forma de tablas (dataframes).
import pandas as pd

In [18]:
# 1. PARÁMETROS DEL REGISTRO
n_channels = 16 # Número de canales del probe
fs = 2500       # Frecuencia de muestreo (Hz)

# 2. CARGAR EL ARCHIVO DE DATOS
carpeta_sesion = Path(r"D:\Violeta\ANALISIS\WT_90\2025_07_16_0000")

archivo_lfp_dat = carpeta_sesion / "LFP_downsampled.dat"
archivo_ripple_analysis = carpeta_sesion / "analyset" / "rippleAnalysis.mat"

# Para comprobar que se encuentran los archivos 
if not archivo_lfp_dat.exists():
    print(f"No se encuentra el archivo .dat en:\n{archivo_lfp_dat}")
elif not archivo_ripple_analysis.exists():
    print(f"No se encuentra el archivo .mat en:\n{archivo_ripple_analysis}")
else:
    print("Archivos encontrados e identificados")
    print(f"Sesión activa: {carpeta_sesion.name}")

'''
# 2.1. CARGAR MÚLTIPLES ARCHIVOS DE DATOS
carpeta_principal = Path(r"D:\Violeta\ANALISIS")
sesiones_validas = []

# rglob() busca en TODAS las subcarpetas archivos que se llamen así
for dat_file in carpeta_principal.rglob("LFP_downsampled.dat"):
    folder_sesion = dat_file.parent
    mat_file = folder_sesion / "analyset" / "rippleAnalysis.mat"
    if mat_file.exists():
        sesiones_validas.append(folder_sesion)
        
print(f"Sesiones válidas encontradas en total: {len(sesiones_validas)}")
        
'''

print(f"Todo listo para analizar. \nFrecuencia: {fs} Hz \nCanales: {n_channels}")

Archivos encontrados e identificados
Sesión activa: 2025_07_16_0000
Todo listo para analizar. 
Frecuencia: 2500 Hz 
Canales: 16


<>:22: SyntaxWarning: "\V" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\V"? A raw string is also an option.
<>:22: SyntaxWarning: "\V" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\V"? A raw string is also an option.
C:\Users\natal\AppData\Local\Temp\ipykernel_25016\3364673232.py:22: SyntaxWarning: "\V" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\V"? A raw string is also an option.
  carpeta_principal = Path(r"D:\Violeta\ANALISIS")


In [13]:
# 3. LEER EL ARCHIVO DE DATOS
# En registros neuronales, el formato estándar de los datos suele ser un tipo de número llamado 'int16' (entero de 16 bits).
datos_brutos = np.fromfile(archivo_actual, dtype=np.int16)

print(f"Puntos totales leídos: {len(datos_brutos)}")

# Cortar la fila en bloques de 16 canales.
# El '-1' es un truco de Python que significa: "calcula tú cuántas filas salen, yo solo te fijo que hay {n_channels} columnas"
datos_matriz = datos_brutos.reshape(-1, n_channels)

# Extraer cuántos instantes de tiempo reales tenemos
puntos_de_tiempo = datos_matriz.shape[0]

print(f"¡Éxito! Ahora hay una matriz con: {puntos_de_tiempo} filas (tiempo) y {datos_matriz.shape[1]} columnas (canales).")

Puntos totales leídos: 36632736
¡Éxito! Ahora hay una matriz con: 2289546 filas (tiempo) y 16 columnas (canales).


In [14]:
# 4. CREAR CARPETA DE RESULTADOS
# .parent da la ruta de la carpeta que contiene el archivo que se está leyendo.
carpeta_sesion = Path(archivo_actual).parent

# Poner nombre a la carpeta de resultados.
carpeta_resultados = carpeta_sesion / "ProcesamientoNatalia"

# Crearla físicamente en el disco duro.
# exist_ok=True significa que si ya existe (porque se ejecuta varias veces), no dará error.
carpeta_resultados.mkdir(exist_ok=True)

print(f"¡Carpeta lista! Todo se guardará en:\n{carpeta_resultados}")

¡Carpeta lista! Todo se guardará en:
D:\Violeta\ANALISIS\WT_90\2025_07_16_0000\ProcesamientoNatalia


In [ ]:
# 5. SELECCIÓN DE LOS RIPPLE DETECTADOS
# Definir la ruta de tu archivo .mat (archivo en la misma carpeta que el .dat)
archivo_mat = carpeta_sesion / "analyset" / "rippleAnalysis.mat"

# 2. Cargamos el contenido del archivo
mat_data = sio.loadmat(archivo_mat)

# 3. Extraemos la estructura rippleAnalysis
# En Python, los structs de MATLAB se leen dentro de un diccionario
ripple_analysis = mat_data['rippleAnalysis']

# Extraemos la variable iRipsMiddle y le restamos 1 para ajustar a Python (base 0)
# Usamos .flatten() para convertirlo en una lista simple de números
picos_ripples = ripple_analysis['iRipsMiddle'][0, 0].flatten() - 1

num_ripples = len(picos_ripples)

print(f"¡Éxito al cargar el archivo .mat!")
print(f"Se han detectado {num_ripples} ripples en esta sesión.")
print(f"El centro del primer ripple está en el punto: {picos_ripples[0]}")